In [1]:
import pandas as pd
from models.netload_reader.get_victron_data import fetch_victron_data, fetch_victron_soc
from models.legal_financial_and_metering.electric_utility_company import ElectricUtilityCompany
from pathlib import Path
from sklearn.metrics import mean_absolute_error

SIM_START = pd.Timestamp('2026-04-10 19:30', tz='Europe/Vienna')
SIM_END = pd.Timestamp('2026-04-17 19:30', tz='Europe/Vienna')
TZ = 'Europe/Vienna'

# Victron-Daten holen (inkl. P_grid = Gc - Pg mit Batterieeffekten)
df_victron = fetch_victron_data(start=SIM_START, end=SIM_END, timezone=TZ)
soc_series = fetch_victron_soc(start=SIM_START, end=SIM_END, timezone=TZ)
df_victron[['net_load', 'P_grid']].head(10)

# Zeitraster
times = pd.date_range(start=SIM_START, end=SIM_END, freq='15min', inclusive='left', tz=TZ)

# MPC-Ergebnisse laden
df_mpc = pd.read_parquet('../data/mp_controller_homebattery/mpc_results_20260410_193659.parquet')

# Simulation laden
output_dir = Path('../data/output/home_bess_simulation/mp_control')
chosen_parquet =  output_dir / '20260503_0949.parquet'
if chosen_parquet is None:
    chosen_parquet = sorted(output_dir.glob('*.parquet'))[-1]
df_sim = pd.read_parquet(chosen_parquet)
print(f"Simulationsdaten: {chosen_parquet.name}")


Fetching VRM data from 2026-04-10 19:30:00+02:00 to 2026-04-17 19:15:00+02:00 ...
Simulationsdaten: 20260503_0949.parquet


In [2]:

# ElectricUtilityCompany mit denselben Parametern wie im Szenario
utility_params = dict(
    use_epex_prices=True,
    download_start=str(SIM_START.date()),
    download_end=str((SIM_END + pd.Timedelta(days=2)).date()),
    epex_offset_buy_eur_kwh=0.0144,
    epex_offset_sell_eur_kwh=0.006,
    grid_fee_eur_kwh=0.06,
    vat=0.20,
    timezone=TZ,
)
utility_meas = ElectricUtilityCompany(name='utility_meas', **utility_params)
utility_sim = ElectricUtilityCompany(name='utility_sim', **utility_params)

# --- Messung ---
P_grid_measured = df_victron['P_grid']
P_grid_set = df_mpc.groupby('mpc_time')['set_netload_kw'].first()
P_grid_set = P_grid_set.resample('15min').mean() * 1000  # kW → W
P_grid_set = P_grid_set[SIM_START:SIM_END]

# Evalute either measured or set values (for now, we focus on measured)
P_grid = P_grid_measured
assert len(P_grid) == len(times), f"{len(P_grid)} vs {len(times)}"
bills_meas = {}
for t, P_el in P_grid.items():
    result = utility_meas.step(t, P_el)
    bills_meas[t] = result['bill']

P_grid_sim = df_sim[('Smart Meter', 'outputs', 'P_grid')]
bills_sim = {}
for t, P_el in P_grid_sim.items():
    result = utility_sim.step(t, P_el)
    bills_sim[t] = result['bill']

# Tagesend-Wert = kumulierte Rechnung am Ende jedes Tages
bill_meas_s = pd.Series(bills_meas)
bill_sim_s = pd.Series(bills_sim)
daily_meas = bill_meas_s.groupby(bill_meas_s.index.date).last()
daily_sim = bill_sim_s.groupby(bill_sim_s.index.date).last()

daily = pd.DataFrame({'Messung (€)': daily_meas.values, 'Simulation (€)': daily_sim.values},
                      index=pd.to_datetime(daily_meas.index))
daily.index.name = 'Datum'

print("\nTägliche Stromkosten:")
print(daily.round(4).to_string())
print(f"\nGesamt:  Messung = {daily['Messung (€)'].sum():.4f} €  |  Simulation = {daily['Simulation (€)'].sum():.4f} €")

# Berechne den MAE zwischen gemessener und simulierter P_grid
mae = mean_absolute_error(P_grid_measured, P_grid_sim)
reference = P_grid_measured.max() - P_grid_measured.min()  # Bereich der Werte als Referenz
normalized_mae = 100 * mae / reference
print(f"\nNormalized Mean Absolute Error (nMAE): {normalized_mae:.4f}%")



Tägliche Stromkosten:
            Messung (€)  Simulation (€)
Datum                                  
2026-04-10      -3.1899         -3.2920
2026-04-11      -1.1463         -1.5539
2026-04-12       0.0599         -0.0247
2026-04-13       1.2555          1.0951
2026-04-14      -3.7437         -3.9278
2026-04-15      -0.2313         -0.5810
2026-04-16      -3.9265         -3.9960
2026-04-17      -3.3030         -3.3944

Gesamt:  Messung = -14.2254 €  |  Simulation = -15.6747 €

Normalized Mean Absolute Error (nMAE): 1.2055%


In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Letzter verfügbarer Forecast pro Zeitschritt (1-step-ahead)
forecast_df = df_mpc.groupby('timestamp').last()
netload_forecast = forecast_df['netload_forecast_kw'] * 1000  # kW → W

# Simulierte Daten extrahieren
P_grid_sim = df_sim[('Smart Meter', 'outputs', 'P_grid')]
soc_sim = df_sim[('Battery', 'outputs', 'SOC_percent')]
P_bat_sim = df_sim[('Battery', 'outputs', 'P_bat')]

# Gemessene Daten
P_bat_meas = df_victron['P_bat']
net_load_meas = df_victron['net_load']
P_grid_meas = df_victron['P_grid']

soc_measured_plot = soc_series.copy()
soc_sim_plot = soc_sim.copy()
soc_measured_plot.index = soc_measured_plot.index.tz_convert(TZ)
soc_sim_plot.index = soc_sim_plot.index.tz_convert(TZ)
soc_sim_aligned = soc_sim_plot.reindex(soc_measured_plot.index)
soc_compare = pd.DataFrame(
    {
        'measured': soc_measured_plot,
        'simulated': soc_sim_aligned,
    }
).dropna()
soc_diff = soc_compare['measured'] - soc_compare['simulated']

# EPEX-Preise
prices_buy = utility_meas._price_buy_eur_kwh
prices_sell = utility_meas._price_sell_eur_kwh

# --- Plotly Figure mit 7 Subplots ---
fig = make_subplots(
    rows=7, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=(
        'SOC',
        'SOC Difference (Measured - Simulation)',
        'P_bat',
        'Net Load',
        'P_grid (Messung vs Simulation)',
        'EPEX-Preise',
        'P_grid (Set vs Measured)',
    ),
)

# 1) SOC
fig.add_trace(go.Scatter(x=soc_measured_plot.index, y=soc_measured_plot.values, mode='lines',
    line=dict(shape='hv', color='orange'), name='SOC (Real MPC)', opacity=0.8), row=1, col=1)
fig.add_trace(go.Scatter(x=soc_sim_plot.index, y=soc_sim_plot.values, mode='lines',
    line=dict(shape='hv', color='green'), name='SOC (Simulation)', opacity=0.8), row=1, col=1)

# 2) SOC-Differenz
fig.add_trace(go.Scatter(
    x=soc_diff.index, y=soc_diff.values, mode='lines',
    line=dict(shape='hv', color='crimson'),
    name='SOC difference (measured - simulated)', opacity=0.85), row=2, col=1)

# 3) P_bat
fig.add_trace(go.Scatter(x=P_bat_meas.index, y=P_bat_meas.values / 1000, mode='lines',
    line=dict(shape='hv', color='purple'), name='P_bat (Real MPC)', opacity=0.8), row=3, col=1)
fig.add_trace(go.Scatter(x=P_bat_sim.index, y=P_bat_sim.values / 1000, mode='lines',
    line=dict(shape='hv', color='cyan'), name='P_bat (Simulation)', opacity=0.8), row=3, col=1)

# 4) Net Load + Forecast
fig.add_trace(go.Scatter(x=net_load_meas.index, y=net_load_meas.values / 1000, mode='lines',
    line=dict(shape='hv', color='brown'), name='net_load (VRM)', opacity=0.8), row=4, col=1)
fig.add_trace(go.Scatter(x=netload_forecast.index, y=netload_forecast.values / 1000, mode='lines',
    line=dict(shape='hv', color='red', dash='dash'), name='net_load Forecast (MPC)', opacity=0.7), row=4, col=1)

# 5) P_grid Messung vs Simulation
fig.add_trace(go.Scatter(x=P_grid_meas.index, y=P_grid_meas.values / 1000, mode='lines',
    line=dict(shape='hv', color='blue'), name='P_grid (Real MPC)', opacity=0.8), row=5, col=1)
fig.add_trace(go.Scatter(x=P_grid_sim.index, y=P_grid_sim.values / 1000, mode='lines',
    line=dict(shape='hv', color='olive'), name='P_grid (Simulation)', opacity=0.8), row=5, col=1)

# 6) EPEX-Preise
fig.add_trace(go.Scatter(x=prices_buy.index, y=prices_buy.values * 100, mode='lines',
    line=dict(shape='hv', color='red'), name='Bezugspreis (ct/kWh)', opacity=0.8), row=6, col=1)
fig.add_trace(go.Scatter(x=prices_sell.index, y=prices_sell.values * 100, mode='lines',
    line=dict(shape='hv', color='green'), name='Einspeisepreis (ct/kWh)', opacity=0.8), row=6, col=1)

# 7) P_grid Set vs Measured
fig.add_trace(go.Scatter(x=P_grid_set.index, y=P_grid_set.values / 1000, mode='lines',
    line=dict(shape='hv', color='orange'), name='P_grid set', opacity=0.8), row=7, col=1)
fig.add_trace(go.Scatter(x=P_grid_measured.index, y=P_grid_measured.values / 1000, mode='lines',
    line=dict(shape='hv', color='blue'), name='P_grid measured', opacity=0.8), row=7, col=1)

# Zero-lines for relevant subplots
for row in [2, 3, 4, 5, 7]:
    fig.add_hline(y=0, line=dict(color='gray', width=0.5, dash='dash'), row=row, col=1)

# Axis labels
fig.update_yaxes(title_text='SOC (%)', row=1, col=1, range=[0, 100])
fig.update_yaxes(title_text='SOC Δ (%)', row=2, col=1)
fig.update_yaxes(title_text='P_bat (kW)', row=3, col=1)
fig.update_yaxes(title_text='net_load (kW)', row=4, col=1)
fig.update_yaxes(title_text='P_grid (kW)', row=5, col=1)
fig.update_yaxes(title_text='Preis (ct/kWh)', row=6, col=1)
fig.update_yaxes(title_text='P_grid (kW)', row=7, col=1)
fig.update_xaxes(title_text='Zeit', row=7, col=1, range=[SIM_START, SIM_END])

fig.update_layout(height=1650, legend=dict(orientation='h', y=-0.02), template='plotly_white')
fig.show()


In [4]:
dt_h = 0.25  # 15 min
common_index = P_grid_set.index.intersection(P_grid_measured.index).intersection(prices_buy.index).intersection(prices_sell.index)
eval_index = common_index[P_grid_set.loc[common_index].eq(0)]
assert len(eval_index) > 0, 'Keine Zeitpunkte mit P_grid_set == 0 gefunden.'

def grid_profit_eur(power_w, buy_price_eur_kwh, sell_price_eur_kwh, dt_h):
    energy_kwh = power_w / 1000 * dt_h
    values = [(-energy * buy) if energy >= 0 else ((-energy) * sell) for energy, buy, sell in zip(energy_kwh, buy_price_eur_kwh, sell_price_eur_kwh)]
    return pd.Series(values, index=power_w.index)

profit_set_total = grid_profit_eur(P_grid_set.loc[eval_index], prices_buy.loc[eval_index], prices_sell.loc[eval_index], dt_h).sum()
profit_measured_total = grid_profit_eur(P_grid_measured.loc[eval_index], prices_buy.loc[eval_index], prices_sell.loc[eval_index], dt_h).sum()
lost_profit_eur = profit_set_total - profit_measured_total

print(f'Anzahl 15-min-Zeitpunkte mit P_grid_set == 0: {len(eval_index)}')
print(f'Anzahl 15-min-Zeitpunkte mit P_grid_set == 0 in Prozent: {len(eval_index) / len(P_grid_set) * 100:.2f}%')
print(f'Total profit set (nur diese Zeitpunkte):     {profit_set_total:.4f} €')
print(f'Total profit measured (gleiche Zeitpunkte): {profit_measured_total:.4f} €')
print(f'Lost profit in diesen Bereichen:            {lost_profit_eur:.4f} €')


Anzahl 15-min-Zeitpunkte mit P_grid_set == 0: 471
Anzahl 15-min-Zeitpunkte mit P_grid_set == 0 in Prozent: 69.99%
Total profit set (nur diese Zeitpunkte):     0.0000 €
Total profit measured (gleiche Zeitpunkte): -1.1263 €
Lost profit in diesen Bereichen:            1.1263 €


In [8]:
p_grid_compare_index = eval_index.intersection(P_grid_sim.index)
assert len(p_grid_compare_index) > 0, 'Keine gemeinsamen Zeitpunkte fuer den nMAE-Vergleich gefunden.'

p_grid_measured_eval = P_grid_measured.loc[p_grid_compare_index]
p_grid_sim_eval = P_grid_sim.loc[p_grid_compare_index]

mae_p_grid = mean_absolute_error(p_grid_measured_eval, p_grid_sim_eval)
reference_p_grid = P_grid_measured.max() - P_grid_measured.min()

assert reference_p_grid != 0, 'Referenzbereich fuer nMAE ist 0.'

normalized_mae_p_grid = 100 * mae_p_grid / reference_p_grid

print(f'Anzahl 15-min-Zeitpunkte fuer nMAE-Auswertung: {len(p_grid_compare_index)}')
print(f'nMAE zwischen gemessenem und simuliertem P_grid (nur Bereiche mit P_grid_set == 0): {normalized_mae_p_grid:.4f}%')


Anzahl 15-min-Zeitpunkte fuer nMAE-Auswertung: 471
nMAE zwischen gemessenem und simuliertem P_grid (nur Bereiche mit P_grid_set == 0): 0.5885%


In [5]:
import numpy as np
from models.battery_storage.battery_storage import BatteryStorage

dt_h = 0.25  # 15 min in Stunden
my_battery = BatteryStorage('dummy')
E_bat_max_kWh = my_battery.E_max / 3_600_000
t0 = SIM_START    # pd.Timestamp('2026-04-10 20:30', tz=TZ)
t_end = SIM_END

# Serien auf (t0, t_end) einschränken
# SOC(t0) ist der Zustand NACH step(t0), daher beginnt die P_bat-Summe bei t0+Δt
mask_meas = (P_bat_meas.index > t0) & (P_bat_meas.index < t_end)
mask_sim = (P_bat_sim.index > t0) & (P_bat_sim.index < t_end)
P_bat_m = P_bat_meas[mask_meas]
P_bat_s = P_bat_sim[mask_sim]
soc_m = soc_series[(soc_series.index >= t0) & (soc_series.index <= t_end)]
soc_s = soc_sim[(soc_sim.index >= t0) & (soc_sim.index <= t_end)]

# --- Messung ---
E_charge_meas_kWh = P_bat_m.clip(lower=0).sum() * dt_h / 1000
E_discharge_meas_kWh = (-P_bat_m).clip(lower=0).sum() * dt_h / 1000
dSOC_meas = soc_m.iloc[-1] - soc_m.iloc[0]  # positiv = netto geladen
dE_soc_meas_kWh = dSOC_meas / 100 * E_bat_max_kWh
E_loss_meas_kWh = E_charge_meas_kWh - E_discharge_meas_kWh - dE_soc_meas_kWh

# --- Simulation ---
E_charge_sim_kWh = P_bat_s.clip(lower=0).sum() * dt_h / 1000
E_discharge_sim_kWh = (-P_bat_s).clip(lower=0).sum() * dt_h / 1000
dSOC_sim = soc_s.iloc[-1] - soc_s.iloc[0]
dE_soc_sim_kWh = dSOC_sim / 100 * E_bat_max_kWh
E_loss_sim_kWh = E_charge_sim_kWh - E_discharge_sim_kWh - dE_soc_sim_kWh

# Round-Trip Efficiency schätzen (Annahme: η_c = η_d)
# Interne Energiebilanz: E_ch · η_c - E_dis / η_d = ΔE_SOC
# Mit η_c = η_d ergibt sich: E_ch · η² - ΔE_SOC · η - E_dis = 0
def estimate_eta_rt(E_ch, E_dis, dE_soc):
    roots = np.roots([E_ch, -dE_soc, -E_dis])
    real_pos = [r.real for r in roots if np.isreal(r) and r.real > 0]
    if not real_pos:
        return np.nan
    eta = real_pos[0]
    return eta ** 2

eta_rt_meas = estimate_eta_rt(E_charge_meas_kWh, E_discharge_meas_kWh, dE_soc_meas_kWh)
eta_rt_sim = estimate_eta_rt(E_charge_sim_kWh, E_discharge_sim_kWh, dE_soc_sim_kWh)

print(f"Batterie-Energiebilanz ({t0} - {t_end})")
print("=" * 55)
print(f"{'':25s} {'Messung':>10s}  {'Simulation':>10s}")
print(f"{'E_charge (kWh)':25s} {E_charge_meas_kWh:10.3f}  {E_charge_sim_kWh:10.3f}")
print(f"{'E_discharge (kWh)':25s} {E_discharge_meas_kWh:10.3f}  {E_discharge_sim_kWh:10.3f}")
# print(f"{'ΔSOC (%)':25s} {dSOC_meas:+10.1f}  {dSOC_sim:+10.1f}")
# print(f"{'ΔE aus SOC (kWh)':25s} {dE_soc_meas_kWh:+10.3f}  {dE_soc_sim_kWh:+10.3f}")
# print(f"{'Verluste (kWh)':25s} {E_loss_meas_kWh:10.3f}  {E_loss_sim_kWh:10.3f}")
print(f"{'η_roundtrip':25s} {eta_rt_meas:10.1%}  {eta_rt_sim:10.1%}")


Batterie-Energiebilanz (2026-04-10 19:30:00+02:00 - 2026-04-17 19:30:00+02:00)
                             Messung  Simulation
E_charge (kWh)               177.109     184.516
E_discharge (kWh)            137.303     144.266
η_roundtrip                    78.6%       78.5%


In [6]:
from scipy.optimize import minimize

# Einheiten konsistent in kWh halten:
soc_measured_kwh = soc_m / 100 * E_bat_max_kWh
P_bat_meas_kw = P_bat_meas / 1000  # W → kW

assert P_bat_meas_kw.index.equals(soc_measured_kwh.index), "Index von P_bat_meas_kw und soc_measured_kwh muss übereinstimmen"

def rmse(values1, values2):
    assert len(values1) == len(values2), "Die beiden Werte-Arrays müssen die gleiche Länge haben"
    diff = np.array(values1) - np.array(values2)
    return np.sqrt(np.mean(diff ** 2))

def batterymodel(soc_prev, P_bat_kw, eta_c, eta_d):
    if P_bat_kw > 0:  # Laden
        dE = P_bat_kw * 0.25 * eta_c    # kWh
    else:              # Entladen
        dE = P_bat_kw * 0.25 / eta_d    # kWh (negativ)
    return soc_prev + dE

def func(params):
    eta_c, eta_d = params
    soc_list = [soc_measured_kwh.iloc[0]]
    for power_kw in P_bat_meas_kw.iloc[:-1]:
        soc_next = batterymodel(soc_list[-1], power_kw, eta_c, eta_d)
        soc_list.append(soc_next)
    return rmse(soc_measured_kwh.values, soc_list)

result = minimize(func, x0=[0.95, 0.95], bounds=[(0.01, 1.0), (0.01, 1.0)], method='L-BFGS-B')
eta_c_fit, eta_d_fit = result.x

print(f"Geschätzter η_charge:    {eta_c_fit*100:.1f}%")
print(f"Geschätzter η_discharge: {eta_d_fit*100:.1f}%")
print(f"Geschätzter η_roundtrip: {eta_c_fit*eta_d_fit*100:.1f}%")
print(f"RMSE SOC (kWh):          {result.fun:.4f}")


Geschätzter η_charge:    86.7%
Geschätzter η_discharge: 90.6%
Geschätzter η_roundtrip: 78.6%
RMSE SOC (kWh):          1.0717
